In [1]:
import json
import random
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel, get_linear_schedule_with_warmup

from sklearn.metrics import f1_score, classification_report

# Data paths
SPLITS_DIR = "/content/drive/MyDrive/emo/emotion_dataset/splits"
OUTPUT_DIR = "/content/drive/MyDrive/emo/finetune/checkpoints"

# Training parameters
EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
MAX_LENGTH = 128
DROPOUT = 0.1
PATIENCE = 2

# Loss weights
W_EMOTION = 1.0
W_TARGET = 1.0
W_INTENSITY = 1.0

# Other settings
SEED = 42
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
CHECKPOINT_EVERY = 1

# ─── CONSTANTS ───────────────────────────────────────────────────────────────

EMOTIONS = ["anger", "sadness", "happiness", "fear", "disgust", "surprise"]
TARGETS  = ["you", "other", "self", "situation", "none"]

EMOTION2ID = {e: i for i, e in enumerate(EMOTIONS)}
TARGET2ID  = {t: i for i, t in enumerate(TARGETS)}
ID2EMOTION = {i: e for e, i in EMOTION2ID.items()}
ID2TARGET  = {i: t for t, i in TARGET2ID.items()}

# ─── SEED ────────────────────────────────────────────────────────────────────

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ─── DATASET ─────────────────────────────────────────────────────────────────

class EmotionDataset(Dataset):
    def __init__(self, samples: list, tokenizer, max_length: int = 128):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        encoding = self.tokenizer(
            s["text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "emotion_label":  torch.tensor(EMOTION2ID[s["emotion"]], dtype=torch.long),
            "target_label":   torch.tensor(TARGET2ID[s["target"]],   dtype=torch.long),
            "intensity":      torch.tensor(float(s["intensity"]),     dtype=torch.float),
        }


def load_split(split_dir: str, split_name: str) -> list:
    """Load a specific split (train/val/test) from JSON file."""
    path = Path(split_dir) / f"{split_name}.json"
    if not path.exists():
        raise FileNotFoundError(f"Split file not found: {path}")

    samples = json.loads(path.read_text())
    print(f"  Loaded {len(samples):>5} samples from {split_name}.json")

    # Ensure all required fields are present
    for s in samples:
        if "text" not in s or "emotion" not in s or "target" not in s or "intensity" not in s:
            raise ValueError(f"Sample missing required fields: {s}")

    return samples

# ─── MODEL ───────────────────────────────────────────────────────────────────

class EmotionMultiHeadModel(nn.Module):
    """
    RoBERTa backbone with 3 heads:
      - emotion_head   : Linear(768 -> 6)
      - target_head    : Linear(768 -> 5)
      - intensity_head : Linear(768 -> 1)  sigmoid output → [0, 1]
    """
    def __init__(self, model_name: str = "roberta-base", dropout: float = 0.1):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        hidden_size  = self.roberta.config.hidden_size  # 768 for roberta-base

        self.dropout        = nn.Dropout(dropout)
        self.emotion_head   = nn.Linear(hidden_size, len(EMOTIONS))
        self.target_head    = nn.Linear(hidden_size, len(TARGETS))
        self.intensity_head = nn.Sequential(
            nn.Linear(hidden_size, 1),
            nn.Sigmoid(),           # clamp output to [0, 1]
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        # [CLS] token representation
        cls = self.dropout(outputs.last_hidden_state[:, 0, :])

        emotion_logits   = self.emotion_head(cls)    # (B, 6)
        target_logits    = self.target_head(cls)     # (B, 5)
        intensity_preds  = self.intensity_head(cls).squeeze(-1)  # (B,)

        return emotion_logits, target_logits, intensity_preds

# ─── LOSS ────────────────────────────────────────────────────────────────────

class MultiTaskLoss(nn.Module):
    """
    Combined loss = w_emotion * CE + w_target * CE + w_intensity * SmoothL1
    """
    def __init__(self, w_emotion=1.0, w_target=1.0, w_intensity=1.0):
        super().__init__()
        self.w_emotion   = w_emotion
        self.w_target    = w_target
        self.w_intensity = w_intensity
        self.ce          = nn.CrossEntropyLoss()
        self.smooth_l1   = nn.SmoothL1Loss()

    def forward(self, emotion_logits, target_logits, intensity_preds,
                emotion_labels, target_labels, intensity_targets):
        loss_emotion   = self.ce(emotion_logits, emotion_labels)
        loss_target    = self.ce(target_logits,  target_labels)
        loss_intensity = self.smooth_l1(intensity_preds, intensity_targets)

        total = (self.w_emotion   * loss_emotion
               + self.w_target    * loss_target
               + self.w_intensity * loss_intensity)

        return total, loss_emotion, loss_target, loss_intensity

# ─── TRAIN EPOCH ─────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, loss_fn, device, epoch):
    model.train()
    total_loss = 0.0
    n_batches  = 0

    progress_bar = tqdm(loader, desc=f"Epoch {epoch} [Train]", leave=False)

    for batch in progress_bar:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        emotion_labels = batch["emotion_label"].to(device)
        target_labels  = batch["target_label"].to(device)
        intensity_tgts = batch["intensity"].to(device)

        optimizer.zero_grad()

        emotion_logits, target_logits, intensity_preds = model(
            input_ids, attention_mask)

        loss, _, _, _ = loss_fn(
            emotion_logits, target_logits, intensity_preds,
            emotion_labels, target_labels, intensity_tgts)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        n_batches  += 1

        # Update progress bar
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    return total_loss / n_batches

# ─── EVALUATE ────────────────────────────────────────────────────────────────

def evaluate(model, loader, loss_fn, device, split_name="Val"):
    model.eval()
    total_loss = 0.0
    n_batches  = 0

    all_emotion_preds  = []
    all_emotion_labels = []
    all_target_preds   = []
    all_target_labels  = []
    all_intensity_preds   = []
    all_intensity_labels  = []

    progress_bar = tqdm(loader, desc=f"{split_name} Evaluation", leave=False)

    with torch.no_grad():
        for batch in progress_bar:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            emotion_labels = batch["emotion_label"].to(device)
            target_labels  = batch["target_label"].to(device)
            intensity_tgts = batch["intensity"].to(device)

            emotion_logits, target_logits, intensity_preds = model(
                input_ids, attention_mask)

            loss, _, _, _ = loss_fn(
                emotion_logits, target_logits, intensity_preds,
                emotion_labels, target_labels, intensity_tgts)

            total_loss += loss.item()
            n_batches  += 1

            all_emotion_preds.extend(emotion_logits.argmax(dim=1).cpu().tolist())
            all_emotion_labels.extend(emotion_labels.cpu().tolist())
            all_target_preds.extend(target_logits.argmax(dim=1).cpu().tolist())
            all_target_labels.extend(target_labels.cpu().tolist())
            all_intensity_preds.extend(intensity_preds.cpu().tolist())
            all_intensity_labels.extend(intensity_tgts.cpu().tolist())

    avg_loss = total_loss / n_batches

    # Metrics
    emotion_acc = sum(p == l for p, l in
                      zip(all_emotion_preds, all_emotion_labels)) / len(all_emotion_labels)
    target_acc  = sum(p == l for p, l in
                      zip(all_target_preds, all_target_labels)) / len(all_target_labels)

    emotion_f1 = f1_score(all_emotion_labels, all_emotion_preds, average="macro", zero_division=0)
    target_f1  = f1_score(all_target_labels,  all_target_preds,  average="macro", zero_division=0)

    intensity_mae = float(np.mean(np.abs(
        np.array(all_intensity_preds) - np.array(all_intensity_labels))))
    intensity_rmse = float(np.sqrt(np.mean(
        (np.array(all_intensity_preds) - np.array(all_intensity_labels)) ** 2)))

    metrics = {
        "loss":          avg_loss,
        "emotion_acc":   emotion_acc,
        "emotion_f1":    emotion_f1,
        "target_acc":    target_acc,
        "target_f1":     target_f1,
        "intensity_mae": intensity_mae,
        "intensity_rmse": intensity_rmse,
    }

    return metrics

# ─── SAVE CHECKPOINT ─────────────────────────────────────────────────────────

def save_checkpoint(model, optimizer, scheduler, epoch, val_metrics,
                    is_best, output_dir, filename):
    """Save model checkpoint."""
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "val_metrics": val_metrics,
    }

    checkpoint_path = Path(output_dir) / filename
    torch.save(checkpoint, checkpoint_path)

    if is_best:
        best_path = Path(output_dir) / "best_model.pt"
        torch.save(checkpoint, best_path)

# ─── MAIN TRAINING LOOP ──────────────────────────────────────────────────────

def train():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*60}")
    print(f"  EMOTION CLASSIFIER TRAINING")
    print(f"{'='*60}")
    print(f"  Device      : {device}")
    print(f"  Splits dir  : {SPLITS_DIR}")
    print(f"  Output dir  : {OUTPUT_DIR}")
    print(f"  Epochs      : {EPOCHS}")
    print(f"  Batch size  : {BATCH_SIZE}")
    print(f"  Learning rate: {LEARNING_RATE}")
    print(f"{'='*60}\n")

    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    # ── Load splits ──────────────────────────────────────────────────────────
    print("Loading data splits...")
    train_samples = load_split(SPLITS_DIR, "train")
    val_samples   = load_split(SPLITS_DIR, "val")
    test_samples  = load_split(SPLITS_DIR, "test")
    print()

    # ── Tokenizer & datasets ───────────────────────────────────────────────
    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print()

    train_ds = EmotionDataset(train_samples, tokenizer, MAX_LENGTH)
    val_ds   = EmotionDataset(val_samples,   tokenizer, MAX_LENGTH)
    test_ds  = EmotionDataset(test_samples,  tokenizer, MAX_LENGTH)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches  : {len(val_loader)}")
    print(f"  Test batches : {len(test_loader)}\n")

    # ── Model ──────────────────────────────────────────────────────────────
    print("Initializing model...")
    model   = EmotionMultiHeadModel("roberta-base", dropout=DROPOUT).to(device)
    loss_fn = MultiTaskLoss(w_emotion=W_EMOTION, w_target=W_TARGET, w_intensity=W_INTENSITY)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params    : {total_params:,}")
    print(f"  Trainable params: {trainable_params:,}\n")

    # ── Optimizer & scheduler ──────────────────────────────────────────────
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=0.01)
    total_steps   = len(train_loader) * EPOCHS
    warmup_steps  = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps)

    # ── Training loop ──────────────────────────────────────────────────────
    best_val_loss   = float("inf")
    patience_count  = 0
    history         = []

    print(f"{'Epoch':<6} {'Train Loss':<12} {'Val Loss':<12} "
          f"{'Emo Acc':<10} {'Tgt Acc':<10} {'Int MAE':<10}")
    print("─" * 62)

    for epoch in range(1, EPOCHS + 1):
        # Train
        train_loss = train_epoch(model, train_loader, optimizer,
                                  scheduler, loss_fn, device, epoch)

        # Validate
        val_metrics = evaluate(model, val_loader, loss_fn, device, "Val")

        # Save to history
        row = {
            "epoch":       epoch,
            "train_loss":  train_loss,
            **val_metrics,
        }
        history.append(row)

        # Print progress
        print(f"{epoch:<6} {train_loss:<12.4f} {val_metrics['loss']:<12.4f} "
              f"{val_metrics['emotion_acc']:<10.3f} "
              f"{val_metrics['target_acc']:<10.3f} "
              f"{val_metrics['intensity_mae']:<10.4f}")

        # Save checkpoint periodically
        if epoch % CHECKPOINT_EVERY == 0:
            checkpoint_name = f"checkpoint_epoch_{epoch}.pt"
            save_checkpoint(model, optimizer, scheduler, epoch, val_metrics,
                          False, OUTPUT_DIR, checkpoint_name)
            print(f"       ✓ Checkpoint saved: {checkpoint_name}")

        # Early stopping check
        if val_metrics["loss"] < best_val_loss:
            best_val_loss  = val_metrics["loss"]
            patience_count = 0
            save_checkpoint(model, optimizer, scheduler, epoch, val_metrics,
                          True, OUTPUT_DIR, "best_model.pt")
            print(f"       ✓ Best model saved (val_loss={best_val_loss:.4f})")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"\n⚠ Early stopping at epoch {epoch} "
                      f"(no improvement for {PATIENCE} epochs)")
                break

    # Save training history
    history_path = Path(OUTPUT_DIR) / "training_history.json"
    history_path.write_text(json.dumps(history, indent=2))
    print(f"\n✓ Training history saved to {history_path}")

    # Save tokenizer
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"✓ Tokenizer saved to {OUTPUT_DIR}")

    print(f"\n{'='*60}")
    print(f"  TRAINING COMPLETE")
    print(f"  Best validation loss: {best_val_loss:.4f}")
    print(f"  Checkpoints saved to: {OUTPUT_DIR}")
    print(f"{'='*60}\n")

# ─── ENTRY POINT ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    train()


  EMOTION CLASSIFIER TRAINING
  Device      : cuda
  Splits dir  : /content/drive/MyDrive/emo/emotion_dataset/splits
  Output dir  : /content/drive/MyDrive/emo/finetune/checkpoints
  Epochs      : 5
  Batch size  : 32
  Learning rate: 2e-05

Loading data splits...
  Loaded 12536 samples from train.json
  Loaded  2637 samples from val.json
  Loaded  2827 samples from test.json

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  Total params    : 124,654,860
  Trainable params: 124,654,860

Epoch  Train Loss   Val Loss     Emo Acc    Tgt Acc    Int MAE   
──────────────────────────────────────────────────────────────


1      1.7314       0.7741       0.934      0.782      0.2033    
       ✓ Checkpoint saved: checkpoint_epoch_1.pt
       ✓ Best model saved (val_loss=0.7741)


2      0.6686       0.6712       0.943      0.804      0.1629    
       ✓ Checkpoint saved: checkpoint_epoch_2.pt
       ✓ Best model saved (val_loss=0.6712)


3      0.4914       0.6810       0.946      0.814      0.1533    
       ✓ Checkpoint saved: checkpoint_epoch_3.pt


4      0.3767       0.6940       0.949      0.811      0.1512    
       ✓ Checkpoint saved: checkpoint_epoch_4.pt

⚠ Early stopping at epoch 4 (no improvement for 2 epochs)

✓ Training history saved to /content/drive/MyDrive/emo/finetune/checkpoints/training_history.json
✓ Tokenizer saved to /content/drive/MyDrive/emo/finetune/checkpoints

  TRAINING COMPLETE
  Best validation loss: 0.6712
  Checkpoints saved to: /content/drive/MyDrive/emo/finetune/checkpoints



In [2]:
import json
import random
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel

from sklearn.metrics import f1_score, classification_report, confusion_matrix

SPLITS_DIR = "/content/drive/MyDrive/emo/emotion_dataset/splits"
MODEL_DIR = "/content/drive/MyDrive/emo/finetune/checkpoints"

BATCH_SIZE = 32
MAX_LENGTH = 128

SEED = 42
SAVE_RESULTS = True
SHOW_CONFUSION_MATRICES = True

# ─── CONSTANTS ───────────────────────────────────────────────────────────────

EMOTIONS = ["anger", "sadness", "happiness", "fear", "disgust", "surprise"]
TARGETS  = ["you", "other", "self", "situation", "none"]

EMOTION2ID = {e: i for i, e in enumerate(EMOTIONS)}
TARGET2ID  = {t: i for i, t in enumerate(TARGETS)}
ID2EMOTION = {i: e for e, i in EMOTION2ID.items()}
ID2TARGET  = {i: t for t, i in TARGET2ID.items()}

# ─── SEED ────────────────────────────────────────────────────────────────────

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ─── DATASET ─────────────────────────────────────────────────────────────────

class EmotionDataset(Dataset):
    def __init__(self, samples: list, tokenizer, max_length: int = 128):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        encoding = self.tokenizer(
            s["text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "emotion_label":  torch.tensor(EMOTION2ID[s["emotion"]], dtype=torch.long),
            "target_label":   torch.tensor(TARGET2ID[s["target"]],   dtype=torch.long),
            "intensity":      torch.tensor(float(s["intensity"]),     dtype=torch.float),
        }


def load_split(split_dir: str, split_name: str) -> list:
    """Load a specific split (train/val/test) from JSON file."""
    path = Path(split_dir) / f"{split_name}.json"
    if not path.exists():
        raise FileNotFoundError(f"Split file not found: {path}")

    samples = json.loads(path.read_text())
    print(f"  Loaded {len(samples):>5} samples from {split_name}.json")
    return samples

# ─── MODEL ───────────────────────────────────────────────────────────────────

class EmotionMultiHeadModel(nn.Module):
    """
    RoBERTa backbone with 3 heads:
      - emotion_head   : Linear(768 -> 6)
      - target_head    : Linear(768 -> 5)
      - intensity_head : Linear(768 -> 1)  sigmoid output → [0, 1]
    """
    def __init__(self, model_name: str = "roberta-base", dropout: float = 0.1):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        hidden_size  = self.roberta.config.hidden_size

        self.dropout        = nn.Dropout(dropout)
        self.emotion_head   = nn.Linear(hidden_size, len(EMOTIONS))
        self.target_head    = nn.Linear(hidden_size, len(TARGETS))
        self.intensity_head = nn.Sequential(
            nn.Linear(hidden_size, 1),
            nn.Sigmoid(),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(outputs.last_hidden_state[:, 0, :])

        emotion_logits   = self.emotion_head(cls)
        target_logits    = self.target_head(cls)
        intensity_preds  = self.intensity_head(cls).squeeze(-1)

        return emotion_logits, target_logits, intensity_preds

# ─── LOSS ────────────────────────────────────────────────────────────────────

class MultiTaskLoss(nn.Module):
    def __init__(self, w_emotion=1.0, w_target=1.0, w_intensity=1.0):
        super().__init__()
        self.w_emotion   = w_emotion
        self.w_target    = w_target
        self.w_intensity = w_intensity
        self.ce          = nn.CrossEntropyLoss()
        self.smooth_l1   = nn.SmoothL1Loss()

    def forward(self, emotion_logits, target_logits, intensity_preds,
                emotion_labels, target_labels, intensity_targets):
        loss_emotion   = self.ce(emotion_logits, emotion_labels)
        loss_target    = self.ce(target_logits,  target_labels)
        loss_intensity = self.smooth_l1(intensity_preds, intensity_targets)

        total = (self.w_emotion   * loss_emotion
               + self.w_target    * loss_target
               + self.w_intensity * loss_intensity)

        return total, loss_emotion, loss_target, loss_intensity

# ─── EVALUATE ────────────────────────────────────────────────────────────────

def evaluate(model, loader, loss_fn, device, detailed: bool = True):
    """Evaluate model on given dataloader."""
    model.eval()
    total_loss = 0.0
    n_batches  = 0

    all_emotion_preds  = []
    all_emotion_labels = []
    all_target_preds   = []
    all_target_labels  = []
    all_intensity_preds   = []
    all_intensity_labels  = []

    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    with torch.no_grad():
        for batch in progress_bar:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            emotion_labels = batch["emotion_label"].to(device)
            target_labels  = batch["target_label"].to(device)
            intensity_tgts = batch["intensity"].to(device)

            emotion_logits, target_logits, intensity_preds = model(
                input_ids, attention_mask)

            loss, _, _, _ = loss_fn(
                emotion_logits, target_logits, intensity_preds,
                emotion_labels, target_labels, intensity_tgts)

            total_loss += loss.item()
            n_batches  += 1

            all_emotion_preds.extend(emotion_logits.argmax(dim=1).cpu().tolist())
            all_emotion_labels.extend(emotion_labels.cpu().tolist())
            all_target_preds.extend(target_logits.argmax(dim=1).cpu().tolist())
            all_target_labels.extend(target_labels.cpu().tolist())
            all_intensity_preds.extend(intensity_preds.cpu().tolist())
            all_intensity_labels.extend(intensity_tgts.cpu().tolist())

    avg_loss = total_loss / n_batches

    # Classification metrics
    emotion_acc = sum(p == l for p, l in
                      zip(all_emotion_preds, all_emotion_labels)) / len(all_emotion_labels)
    target_acc  = sum(p == l for p, l in
                      zip(all_target_preds, all_target_labels)) / len(all_target_labels)

    emotion_f1 = f1_score(all_emotion_labels, all_emotion_preds, average="macro", zero_division=0)
    target_f1  = f1_score(all_target_labels,  all_target_preds,  average="macro", zero_division=0)

    # Per-class F1 scores
    emotion_f1_per_class = f1_score(all_emotion_labels, all_emotion_preds,
                                    average=None, labels=list(range(len(EMOTIONS))), zero_division=0)
    target_f1_per_class = f1_score(all_target_labels, all_target_preds,
                                   average=None, labels=list(range(len(TARGETS))), zero_division=0)

    # Regression metrics
    intensity_mae = float(np.mean(np.abs(
        np.array(all_intensity_preds) - np.array(all_intensity_labels))))
    intensity_mse = float(np.mean(
        (np.array(all_intensity_preds) - np.array(all_intensity_labels)) ** 2))
    intensity_rmse = float(np.sqrt(intensity_mse))

    metrics = {
        "loss": avg_loss,
        "emotion_acc": emotion_acc,
        "emotion_f1": emotion_f1,
        "emotion_f1_per_class": emotion_f1_per_class.tolist(),
        "target_acc": target_acc,
        "target_f1": target_f1,
        "target_f1_per_class": target_f1_per_class.tolist(),
        "intensity_mae": intensity_mae,
        "intensity_mse": intensity_mse,
        "intensity_rmse": intensity_rmse,
    }

    if detailed:
        print("\n" + "="*70)
        print("  EMOTION CLASSIFICATION REPORT")
        print("="*70)
        print(classification_report(
            all_emotion_labels, all_emotion_preds,
            target_names=EMOTIONS, digits=4))

        print("\n" + "="*70)
        print("  TARGET CLASSIFICATION REPORT")
        print("="*70)
        print(classification_report(
            all_target_labels, all_target_preds,
            target_names=TARGETS, digits=4))

        print("\n" + "="*70)
        print("  INTENSITY REGRESSION METRICS")
        print("="*70)
        print(f"  Mean Absolute Error (MAE) : {intensity_mae:.4f}")
        print(f"  Mean Squared Error (MSE)  : {intensity_mse:.4f}")
        print(f"  Root Mean Squared Error   : {intensity_rmse:.4f}")

        # Show intensity prediction distribution
        print(f"\n  Intensity Stats:")
        print(f"    True range    : [{min(all_intensity_labels):.3f}, {max(all_intensity_labels):.3f}]")
        print(f"    Pred range    : [{min(all_intensity_preds):.3f}, {max(all_intensity_preds):.3f}]")
        print(f"    True mean     : {np.mean(all_intensity_labels):.3f}")
        print(f"    Pred mean     : {np.mean(all_intensity_preds):.3f}")

        if SHOW_CONFUSION_MATRICES:
            print("\n" + "="*70)
            print("  CONFUSION MATRICES")
            print("="*70)

            # Emotion confusion matrix
            emotion_cm = confusion_matrix(all_emotion_labels, all_emotion_preds)
            print("\n  Emotion Confusion Matrix (rows=true, cols=pred):")
            print("  " + " ".join(f"{e[:4]:>5}" for e in EMOTIONS))
            for i, row in enumerate(emotion_cm):
                print(f"  {EMOTIONS[i][:4]:>5} " + " ".join(f"{x:5d}" for x in row))

            # Target confusion matrix
            target_cm = confusion_matrix(all_target_labels, all_target_preds)
            print("\n  Target Confusion Matrix (rows=true, cols=pred):")
            print("  " + " ".join(f"{t[:4]:>5}" for t in TARGETS))
            for i, row in enumerate(target_cm):
                print(f"  {TARGETS[i][:4]:>5} " + " ".join(f"{x:5d}" for x in row))

    return metrics, (all_emotion_preds, all_emotion_labels, all_target_preds, all_target_labels,
                     all_intensity_preds, all_intensity_labels)

# ─── MAIN TEST FUNCTION ──────────────────────────────────────────────────────

def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"\n{'='*70}")
    print(f"  EMOTION CLASSIFIER TESTING")
    print(f"{'='*70}")
    print(f"  Device      : {device}")
    print(f"  Splits dir  : {SPLITS_DIR}")
    print(f"  Model dir   : {MODEL_DIR}")
    print(f"{'='*70}\n")

    # Load test data
    print("Loading test data...")
    test_samples = load_split(SPLITS_DIR, "test")
    print()

    # Load tokenizer
    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained(MODEL_DIR)
    print()

    # Load model
    print("Loading model...")
    model = EmotionMultiHeadModel("roberta-base", dropout=0.1).to(device)

    checkpoint_path = Path(MODEL_DIR) / "best_model.pt"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Handle different checkpoint formats
    if "model_state" in checkpoint:
        model.load_state_dict(checkpoint["model_state"])
        epoch = checkpoint.get("epoch", "unknown")
        val_metrics = checkpoint.get("val_metrics", {})
    elif "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        epoch = checkpoint.get("epoch", "unknown")
        val_metrics = checkpoint.get("val_metrics", {})
    else:
        model.load_state_dict(checkpoint)
        epoch = "unknown"
        val_metrics = {}

    print(f"  Loaded checkpoint from epoch {epoch}")
    if val_metrics:
        print(f"  Validation metrics at save:")
        print(f"    Loss      : {val_metrics.get('loss', 'N/A'):.4f}")
        print(f"    Emotion F1: {val_metrics.get('emotion_f1', 'N/A'):.3f}")
        print(f"    Target F1 : {val_metrics.get('target_f1', 'N/A'):.3f}")
        print(f"    Intensity MAE: {val_metrics.get('intensity_mae', 'N/A'):.4f}")
    print()

    # Create test dataset and loader
    test_ds = EmotionDataset(test_samples, tokenizer, MAX_LENGTH)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=2)

    # Evaluate
    loss_fn = MultiTaskLoss()
    print("Running evaluation on test set...")
    test_metrics, predictions = evaluate(model, test_loader, loss_fn, device, detailed=True)

    # Print summary
    print("\n" + "="*70)
    print("  TEST SET SUMMARY")
    print("="*70)
    print(f"  Total samples tested: {len(test_samples)}")
    print(f"  Loss                : {test_metrics['loss']:.4f}")
    print(f"  Emotion Accuracy    : {test_metrics['emotion_acc']:.3f} ({test_metrics['emotion_acc']*100:.1f}%)")
    print(f"  Emotion Macro F1    : {test_metrics['emotion_f1']:.3f}")
    print(f"  Target Accuracy     : {test_metrics['target_acc']:.3f} ({test_metrics['target_acc']*100:.1f}%)")
    print(f"  Target Macro F1     : {test_metrics['target_f1']:.3f}")
    print(f"  Intensity MAE       : {test_metrics['intensity_mae']:.4f}")
    print(f"  Intensity RMSE      : {test_metrics['intensity_rmse']:.4f}")

    # Per-class F1 scores
    print("\n  Per-class Emotion F1 Scores:")
    for i, f1 in enumerate(test_metrics['emotion_f1_per_class']):
        print(f"    {EMOTIONS[i]:12s}: {f1:.4f}")

    print("\n  Per-class Target F1 Scores:")
    for i, f1 in enumerate(test_metrics['target_f1_per_class']):
        print(f"    {TARGETS[i]:12s}: {f1:.4f}")

    print("="*70)

    # Save results
    if SAVE_RESULTS:
        results_path = Path(MODEL_DIR) / "test_results.json"

        # Convert numpy arrays to lists for JSON serialization
        full_results = {
            "test_metrics": {
                k: v for k, v in test_metrics.items()
                if not k.endswith("_per_class")
            },
            "per_class_metrics": {
                "emotion_f1_per_class": {
                    EMOTIONS[i]: test_metrics['emotion_f1_per_class'][i]
                    for i in range(len(EMOTIONS))
                },
                "target_f1_per_class": {
                    TARGETS[i]: test_metrics['target_f1_per_class'][i]
                    for i in range(len(TARGETS))
                }
            },
            "checkpoint_info": {
                "epoch": epoch,
                "validation_metrics_at_save": val_metrics
            },
            "test_config": {
                "splits_dir": str(SPLITS_DIR),
                "model_dir": str(MODEL_DIR),
                "batch_size": BATCH_SIZE,
                "max_length": MAX_LENGTH,
                "num_test_samples": len(test_samples)
            }
        }

        results_path.write_text(json.dumps(full_results, indent=2))
        print(f"\n✓ Results saved to {results_path}")

if __name__ == "__main__":
    main()


  EMOTION CLASSIFIER TESTING
  Device      : cuda
  Splits dir  : /content/drive/MyDrive/emo/emotion_dataset/splits
  Model dir   : /content/drive/MyDrive/emo/finetune/checkpoints

Loading test data...
  Loaded  2827 samples from test.json

Loading tokenizer...

Loading model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded checkpoint from epoch 2
  Validation metrics at save:
    Loss      : 0.6712
    Emotion F1: 0.943
    Target F1 : 0.801
    Intensity MAE: 0.1629

Running evaluation on test set...



  EMOTION CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger     0.8939    0.8806    0.8872       469
     sadness     0.8838    0.9403    0.9112       469
   happiness     0.9814    0.9916    0.9865       479
        fear     0.9679    0.9658    0.9668       468
     disgust     0.9349    0.9151    0.9249       471
    surprise     0.9868    0.9512    0.9686       471

    accuracy                         0.9409      2827
   macro avg     0.9415    0.9408    0.9409      2827
weighted avg     0.9416    0.9409    0.9410      2827


  TARGET CLASSIFICATION REPORT
              precision    recall  f1-score   support

         you     0.9539    0.9199    0.9366       562
       other     0.9299    0.9527    0.9412       571
        self     0.8685    0.8964    0.8822       560
   situation     0.6635    0.7478    0.7032       567
        none     0.7039    0.5996    0.6476       567

    accuracy                         0.8231      2827
   macro avg

In [3]:
import json
import random
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel, get_linear_schedule_with_warmup

from sklearn.metrics import f1_score, classification_report

# Data paths
SPLITS_DIR = "/content/drive/MyDrive/emo/emotion_data/splits"
OUTPUT_DIR = "/content/drive/MyDrive/emo/finetune/checkpoints-2"

# Training parameters
EPOCHS        = 8
BATCH_SIZE    = 32
LEARNING_RATE = 2e-5
MAX_LENGTH    = 128
DROPOUT       = 0.1
PATIENCE      = 3

# Loss weights
W_EMOTION   = 1.0
W_TARGET    = 1.0
W_INTENSITY = 3.0

# Other settings
SEED            = 42
WARMUP_RATIO    = 0.1
MAX_GRAD_NORM   = 1.0
CHECKPOINT_EVERY = 1

# ─── CONSTANTS ───────────────────────────────────────────────────────────────

EMOTIONS = ["anger", "sadness", "happiness", "fear", "disgust", "surprise"]
TARGETS  = ["you", "other", "self", "situation"]

EMOTION2ID = {e: i for i, e in enumerate(EMOTIONS)}
TARGET2ID  = {t: i for i, t in enumerate(TARGETS)}
ID2EMOTION = {i: e for e, i in EMOTION2ID.items()}
ID2TARGET  = {i: t for t, i in TARGET2ID.items()}

# ─── SEED ────────────────────────────────────────────────────────────────────

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ─── DATASET ─────────────────────────────────────────────────────────────────

class EmotionDataset(Dataset):
    def __init__(self, samples: list, tokenizer, max_length: int = 128):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        encoding = self.tokenizer(
            s["text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "emotion_label":  torch.tensor(EMOTION2ID[s["emotion"]], dtype=torch.long),
            "target_label":   torch.tensor(TARGET2ID[s["target"]],   dtype=torch.long),
            "intensity":      torch.tensor(float(s["intensity"]),     dtype=torch.float),
        }


def load_split(split_dir: str, split_name: str) -> list:
    path = Path(split_dir) / f"{split_name}.json"
    if not path.exists():
        raise FileNotFoundError(f"Split file not found: {path}")

    samples = json.loads(path.read_text())

    # Filter out any samples with targets not in our 4-class set
    before = len(samples)
    samples = [s for s in samples if s["target"] in TARGET2ID]
    after   = len(samples)
    if before != after:
        print(f"  Filtered {before - after} samples with unknown targets")

    print(f"  Loaded {len(samples):>5} samples from {split_name}.json")

    for s in samples:
        if "text" not in s or "emotion" not in s or "target" not in s or "intensity" not in s:
            raise ValueError(f"Sample missing required fields: {s}")

    return samples

# ─── MODEL ───────────────────────────────────────────────────────────────────

class EmotionMultiHeadModel(nn.Module):
    """
    RoBERTa backbone with 3 heads:
      - emotion_head   : Linear(768 -> 6)
      - target_head    : Linear(768 -> 4)
      - intensity_head : Linear(768 -> 1) + clamp [was Sigmoid]
    """
    def __init__(self, model_name: str = "roberta-base", dropout: float = 0.1):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        hidden_size  = self.roberta.config.hidden_size  # 768

        self.dropout        = nn.Dropout(dropout)
        self.emotion_head   = nn.Linear(hidden_size, len(EMOTIONS))
        self.target_head    = nn.Linear(hidden_size, len(TARGETS))
        self.intensity_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(outputs.last_hidden_state[:, 0, :])

        emotion_logits  = self.emotion_head(cls)                          # (B, 6)
        target_logits   = self.target_head(cls)                           # (B, 4)
        intensity_preds = self.intensity_head(cls).squeeze(-1)            # (B,)
        intensity_preds = torch.clamp(intensity_preds, min=0.0, max=1.0) # keep in range

        return emotion_logits, target_logits, intensity_preds

# ─── LOSS ────────────────────────────────────────────────────────────────────

class MultiTaskLoss(nn.Module):
    """
    Combined loss = w_emotion * CE + w_target * CE + w_intensity * SmoothL1
    w_intensity raised to 3.0 so regression gets equal learning priority
    against the two classification heads.
    """
    def __init__(self, w_emotion=1.0, w_target=1.0, w_intensity=3.0):
        super().__init__()
        self.w_emotion   = w_emotion
        self.w_target    = w_target
        self.w_intensity = w_intensity
        self.ce          = nn.CrossEntropyLoss()
        self.smooth_l1   = nn.SmoothL1Loss()

    def forward(self, emotion_logits, target_logits, intensity_preds,
                emotion_labels, target_labels, intensity_targets):
        loss_emotion   = self.ce(emotion_logits, emotion_labels)
        loss_target    = self.ce(target_logits,  target_labels)
        loss_intensity = self.smooth_l1(intensity_preds, intensity_targets)

        total = (self.w_emotion   * loss_emotion
               + self.w_target    * loss_target
               + self.w_intensity * loss_intensity)

        return total, loss_emotion, loss_target, loss_intensity

# ─── TRAIN EPOCH ─────────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, loss_fn, device, epoch):
    model.train()
    total_loss = 0.0
    n_batches  = 0

    progress_bar = tqdm(loader, desc=f"Epoch {epoch} [Train]", leave=False)

    for batch in progress_bar:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        emotion_labels = batch["emotion_label"].to(device)
        target_labels  = batch["target_label"].to(device)
        intensity_tgts = batch["intensity"].to(device)

        optimizer.zero_grad()

        emotion_logits, target_logits, intensity_preds = model(
            input_ids, attention_mask)

        loss, _, _, _ = loss_fn(
            emotion_logits, target_logits, intensity_preds,
            emotion_labels, target_labels, intensity_tgts)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        n_batches  += 1

        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    return total_loss / n_batches

# ─── EVALUATE ────────────────────────────────────────────────────────────────

def evaluate(model, loader, loss_fn, device, split_name="Val"):
    model.eval()
    total_loss = 0.0
    n_batches  = 0

    all_emotion_preds   = []
    all_emotion_labels  = []
    all_target_preds    = []
    all_target_labels   = []
    all_intensity_preds  = []
    all_intensity_labels = []

    progress_bar = tqdm(loader, desc=f"{split_name} Evaluation", leave=False)

    with torch.no_grad():
        for batch in progress_bar:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            emotion_labels = batch["emotion_label"].to(device)
            target_labels  = batch["target_label"].to(device)
            intensity_tgts = batch["intensity"].to(device)

            emotion_logits, target_logits, intensity_preds = model(
                input_ids, attention_mask)

            loss, _, _, _ = loss_fn(
                emotion_logits, target_logits, intensity_preds,
                emotion_labels, target_labels, intensity_tgts)

            total_loss += loss.item()
            n_batches  += 1

            all_emotion_preds.extend(emotion_logits.argmax(dim=1).cpu().tolist())
            all_emotion_labels.extend(emotion_labels.cpu().tolist())
            all_target_preds.extend(target_logits.argmax(dim=1).cpu().tolist())
            all_target_labels.extend(target_labels.cpu().tolist())
            all_intensity_preds.extend(intensity_preds.cpu().tolist())
            all_intensity_labels.extend(intensity_tgts.cpu().tolist())

    avg_loss = total_loss / n_batches

    emotion_acc = sum(p == l for p, l in
                      zip(all_emotion_preds, all_emotion_labels)) / len(all_emotion_labels)
    target_acc  = sum(p == l for p, l in
                      zip(all_target_preds, all_target_labels)) / len(all_target_labels)

    emotion_f1 = f1_score(all_emotion_labels, all_emotion_preds,
                           average="macro", zero_division=0)
    target_f1  = f1_score(all_target_labels,  all_target_preds,
                           average="macro", zero_division=0)

    intensity_mae  = float(np.mean(np.abs(
        np.array(all_intensity_preds) - np.array(all_intensity_labels))))
    intensity_rmse = float(np.sqrt(np.mean(
        (np.array(all_intensity_preds) - np.array(all_intensity_labels)) ** 2)))

    metrics = {
        "loss":           avg_loss,
        "emotion_acc":    emotion_acc,
        "emotion_f1":     emotion_f1,
        "target_acc":     target_acc,
        "target_f1":      target_f1,
        "intensity_mae":  intensity_mae,
        "intensity_rmse": intensity_rmse,
    }

    return metrics

# ─── SAVE CHECKPOINT ─────────────────────────────────────────────────────────

def save_checkpoint(model, optimizer, scheduler, epoch, val_metrics,
                    is_best, output_dir, filename):
    checkpoint = {
        "epoch":                epoch,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "val_metrics":          val_metrics,
    }
    torch.save(checkpoint, Path(output_dir) / filename)
    if is_best:
        torch.save(checkpoint, Path(output_dir) / "best_model.pt")

# ─── MAIN TRAINING LOOP ──────────────────────────────────────────────────────

def train():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*60}")
    print(f"  EMOTION CLASSIFIER TRAINING")
    print(f"{'='*60}")
    print(f"  Device       : {device}")
    print(f"  Splits dir   : {SPLITS_DIR}")
    print(f"  Output dir   : {OUTPUT_DIR}")
    print(f"  Epochs       : {EPOCHS}")
    print(f"  Batch size   : {BATCH_SIZE}")
    print(f"  Learning rate: {LEARNING_RATE}")
    print(f"  Targets      : {TARGETS}")
    print(f"  Loss weights : emotion={W_EMOTION} target={W_TARGET} intensity={W_INTENSITY}")
    print(f"{'='*60}\n")

    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    print("Loading data splits...")
    train_samples = load_split(SPLITS_DIR, "train")
    val_samples   = load_split(SPLITS_DIR, "val")
    test_samples  = load_split(SPLITS_DIR, "test")
    print()

    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    print()

    train_ds = EmotionDataset(train_samples, tokenizer, MAX_LENGTH)
    val_ds   = EmotionDataset(val_samples,   tokenizer, MAX_LENGTH)
    test_ds  = EmotionDataset(test_samples,  tokenizer, MAX_LENGTH)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches  : {len(val_loader)}")
    print(f"  Test batches : {len(test_loader)}\n")

    print("Initializing model...")
    model   = EmotionMultiHeadModel("roberta-base", dropout=DROPOUT).to(device)
    loss_fn = MultiTaskLoss(w_emotion=W_EMOTION, w_target=W_TARGET,
                            w_intensity=W_INTENSITY)

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params    : {total_params:,}")
    print(f"  Trainable params: {trainable_params:,}\n")

    optimizer    = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                     weight_decay=0.01)
    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps)

    best_val_loss  = float("inf")
    patience_count = 0
    history        = []

    print(f"{'Epoch':<6} {'Train Loss':<12} {'Val Loss':<12} "
          f"{'Emo Acc':<10} {'Tgt Acc':<10} {'Int MAE':<10}")
    print("─" * 62)

    for epoch in range(1, EPOCHS + 1):
        train_loss  = train_epoch(model, train_loader, optimizer,
                                   scheduler, loss_fn, device, epoch)
        val_metrics = evaluate(model, val_loader, loss_fn, device, "Val")

        history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

        print(f"{epoch:<6} {train_loss:<12.4f} {val_metrics['loss']:<12.4f} "
              f"{val_metrics['emotion_acc']:<10.3f} "
              f"{val_metrics['target_acc']:<10.3f} "
              f"{val_metrics['intensity_mae']:<10.4f}")

        if epoch % CHECKPOINT_EVERY == 0:
            save_checkpoint(model, optimizer, scheduler, epoch, val_metrics,
                            False, OUTPUT_DIR, f"checkpoint_epoch_{epoch}.pt")
            print(f"       ✓ Checkpoint saved: checkpoint_epoch_{epoch}.pt")

        if val_metrics["loss"] < best_val_loss:
            best_val_loss  = val_metrics["loss"]
            patience_count = 0
            save_checkpoint(model, optimizer, scheduler, epoch, val_metrics,
                            True, OUTPUT_DIR, "best_model.pt")
            print(f"       ✓ Best model saved (val_loss={best_val_loss:.4f})")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"\n⚠ Early stopping at epoch {epoch} "
                      f"(no improvement for {PATIENCE} epochs)")
                break

    history_path = Path(OUTPUT_DIR) / "training_history.json"
    history_path.write_text(json.dumps(history, indent=2))
    print(f"\n✓ Training history saved to {history_path}")

    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"✓ Tokenizer saved to {OUTPUT_DIR}")

    print(f"\n{'='*60}")
    print(f"  TRAINING COMPLETE")
    print(f"  Best validation loss: {best_val_loss:.4f}")
    print(f"  Checkpoints saved to: {OUTPUT_DIR}")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    train()


  EMOTION CLASSIFIER TRAINING
  Device       : cuda
  Splits dir   : /content/drive/MyDrive/emo/emotion_data/splits
  Output dir   : /content/drive/MyDrive/emo/finetune/checkpoints-2
  Epochs       : 8
  Batch size   : 32
  Learning rate: 2e-05
  Targets      : ['you', 'other', 'self', 'situation']
  Loss weights : emotion=1.0 target=1.0 intensity=3.0

Loading data splits...
  Loaded 16727 samples from train.json
  Loaded  3532 samples from val.json
  Loaded  3741 samples from test.json

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

1      1.5884       0.5004       0.930      0.911      0.1692    
       ✓ Checkpoint saved: checkpoint_epoch_1.pt
       ✓ Best model saved (val_loss=0.5004)


2      0.4531       0.3888       0.947      0.931      0.1551    
       ✓ Checkpoint saved: checkpoint_epoch_2.pt
       ✓ Best model saved (val_loss=0.3888)


3      0.3103       0.4092       0.955      0.921      0.1499    
       ✓ Checkpoint saved: checkpoint_epoch_3.pt


4      0.2233       0.4066       0.951      0.934      0.1349    
       ✓ Checkpoint saved: checkpoint_epoch_4.pt


5      0.1623       0.4277       0.956      0.937      0.1454    
       ✓ Checkpoint saved: checkpoint_epoch_5.pt

⚠ Early stopping at epoch 5 (no improvement for 3 epochs)

✓ Training history saved to /content/drive/MyDrive/emo/finetune/checkpoints-2/training_history.json
✓ Tokenizer saved to /content/drive/MyDrive/emo/finetune/checkpoints-2

  TRAINING COMPLETE
  Best validation loss: 0.3888
  Checkpoints saved to: /content/drive/MyDrive/emo/finetune/checkpoints-2



In [7]:
import json
import random
import numpy as np
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel

from sklearn.metrics import f1_score, classification_report, confusion_matrix

SPLITS_DIR = "/content/drive/MyDrive/emo/emotion_data/splits"
MODEL_DIR  = "/content/drive/MyDrive/emo/finetune/checkpoints-2"

BATCH_SIZE             = 32
MAX_LENGTH             = 128
SEED                   = 42
SAVE_RESULTS           = True
SHOW_CONFUSION_MATRICES = True

# ─── CONSTANTS — must match training exactly ──────────────────────────────────

EMOTIONS = ["anger", "sadness", "happiness", "fear", "disgust", "surprise"]
TARGETS  = ["you", "other", "self", "situation"]   # none removed

EMOTION2ID = {e: i for i, e in enumerate(EMOTIONS)}
TARGET2ID  = {t: i for i, t in enumerate(TARGETS)}
ID2EMOTION = {i: e for e, i in EMOTION2ID.items()}
ID2TARGET  = {i: t for t, i in TARGET2ID.items()}

# ─── SEED ────────────────────────────────────────────────────────────────────

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ─── DATASET ─────────────────────────────────────────────────────────────────

class EmotionDataset(Dataset):
    def __init__(self, samples: list, tokenizer, max_length: int = 128):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        encoding = self.tokenizer(
            s["text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "emotion_label":  torch.tensor(EMOTION2ID[s["emotion"]], dtype=torch.long),
            "target_label":   torch.tensor(TARGET2ID[s["target"]],   dtype=torch.long),
            "intensity":      torch.tensor(float(s["intensity"]),     dtype=torch.float),
        }


def load_split(split_dir: str, split_name: str) -> list:
    path = Path(split_dir) / f"{split_name}.json"
    if not path.exists():
        raise FileNotFoundError(f"Split file not found: {path}")

    samples = json.loads(path.read_text())

    # Drop any samples with targets not in the 4-class set (e.g. old 'none' samples)
    before  = len(samples)
    samples = [s for s in samples if s["target"] in TARGET2ID]
    dropped = before - len(samples)
    if dropped:
        print(f"  Filtered {dropped} samples with unknown targets")

    print(f"  Loaded {len(samples):>5} samples from {split_name}.json")
    return samples

# ─── MODEL — must match training architecture exactly ─────────────────────────

class EmotionMultiHeadModel(nn.Module):
    """
    RoBERTa backbone with 3 heads:
      - emotion_head   : Linear(768 -> 6)
      - target_head    : Linear(768 -> 4)   [was 5]
      - intensity_head : Linear(768 -> 1) + clamp  [was Sigmoid]
    """
    def __init__(self, model_name: str = "roberta-base", dropout: float = 0.1):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        hidden_size  = self.roberta.config.hidden_size

        self.dropout        = nn.Dropout(dropout)
        self.emotion_head   = nn.Linear(hidden_size, len(EMOTIONS))
        self.target_head    = nn.Linear(hidden_size, len(TARGETS))  # 4 outputs
        self.intensity_head = nn.Linear(hidden_size, 1)             # no Sigmoid

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(outputs.last_hidden_state[:, 0, :])

        emotion_logits  = self.emotion_head(cls)
        target_logits   = self.target_head(cls)
        intensity_preds = self.intensity_head(cls).squeeze(-1)
        intensity_preds = torch.clamp(intensity_preds, min=0.0, max=1.0)

        return emotion_logits, target_logits, intensity_preds

# ─── LOSS ────────────────────────────────────────────────────────────────────

class MultiTaskLoss(nn.Module):
    def __init__(self, w_emotion=1.0, w_target=1.0, w_intensity=3.0):
        super().__init__()
        self.w_emotion   = w_emotion
        self.w_target    = w_target
        self.w_intensity = w_intensity
        self.ce          = nn.CrossEntropyLoss()
        self.smooth_l1   = nn.SmoothL1Loss()

    def forward(self, emotion_logits, target_logits, intensity_preds,
                emotion_labels, target_labels, intensity_targets):
        loss_emotion   = self.ce(emotion_logits, emotion_labels)
        loss_target    = self.ce(target_logits,  target_labels)
        loss_intensity = self.smooth_l1(intensity_preds, intensity_targets)
        total = (self.w_emotion   * loss_emotion
               + self.w_target    * loss_target
               + self.w_intensity * loss_intensity)
        return total, loss_emotion, loss_target, loss_intensity

# ─── EVALUATE ────────────────────────────────────────────────────────────────

def evaluate(model, loader, loss_fn, device, detailed: bool = True):
    model.eval()
    total_loss = 0.0
    n_batches  = 0

    all_emotion_preds    = []
    all_emotion_labels   = []
    all_target_preds     = []
    all_target_labels    = []
    all_intensity_preds  = []
    all_intensity_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            emotion_labels = batch["emotion_label"].to(device)
            target_labels  = batch["target_label"].to(device)
            intensity_tgts = batch["intensity"].to(device)

            emotion_logits, target_logits, intensity_preds = model(
                input_ids, attention_mask)

            loss, _, _, _ = loss_fn(
                emotion_logits, target_logits, intensity_preds,
                emotion_labels, target_labels, intensity_tgts)

            total_loss += loss.item()
            n_batches  += 1

            all_emotion_preds.extend(emotion_logits.argmax(dim=1).cpu().tolist())
            all_emotion_labels.extend(emotion_labels.cpu().tolist())
            all_target_preds.extend(target_logits.argmax(dim=1).cpu().tolist())
            all_target_labels.extend(target_labels.cpu().tolist())
            all_intensity_preds.extend(intensity_preds.cpu().tolist())
            all_intensity_labels.extend(intensity_tgts.cpu().tolist())

    avg_loss = total_loss / n_batches

    emotion_acc = sum(p == l for p, l in
                      zip(all_emotion_preds, all_emotion_labels)) / len(all_emotion_labels)
    target_acc  = sum(p == l for p, l in
                      zip(all_target_preds, all_target_labels)) / len(all_target_labels)

    emotion_f1 = f1_score(all_emotion_labels, all_emotion_preds,
                           average="macro", zero_division=0)
    target_f1  = f1_score(all_target_labels, all_target_preds,
                           average="macro", zero_division=0)

    emotion_f1_per_class = f1_score(all_emotion_labels, all_emotion_preds,
                                    average=None,
                                    labels=list(range(len(EMOTIONS))),
                                    zero_division=0)
    target_f1_per_class  = f1_score(all_target_labels, all_target_preds,
                                    average=None,
                                    labels=list(range(len(TARGETS))),
                                    zero_division=0)

    intensity_arr  = np.array(all_intensity_preds)
    intensity_true = np.array(all_intensity_labels)
    intensity_mae  = float(np.mean(np.abs(intensity_arr - intensity_true)))
    intensity_mse  = float(np.mean((intensity_arr - intensity_true) ** 2))
    intensity_rmse = float(np.sqrt(intensity_mse))

    metrics = {
        "loss":                  avg_loss,
        "emotion_acc":           emotion_acc,
        "emotion_f1":            emotion_f1,
        "emotion_f1_per_class":  emotion_f1_per_class.tolist(),
        "target_acc":            target_acc,
        "target_f1":             target_f1,
        "target_f1_per_class":   target_f1_per_class.tolist(),
        "intensity_mae":         intensity_mae,
        "intensity_mse":         intensity_mse,
        "intensity_rmse":        intensity_rmse,
    }

    if detailed:
        print("\n" + "=" * 70)
        print("  EMOTION CLASSIFICATION REPORT")
        print("=" * 70)
        print(classification_report(
            all_emotion_labels, all_emotion_preds,
            target_names=EMOTIONS, digits=4))

        print("\n" + "=" * 70)
        print("  TARGET CLASSIFICATION REPORT")
        print("=" * 70)
        print(classification_report(
            all_target_labels, all_target_preds,
            target_names=TARGETS, digits=4))

        print("\n" + "=" * 70)
        print("  INTENSITY REGRESSION METRICS")
        print("=" * 70)
        print(f"  Mean Absolute Error (MAE) : {intensity_mae:.4f}")
        print(f"  Mean Squared Error (MSE)  : {intensity_mse:.4f}")
        print(f"  Root Mean Squared Error   : {intensity_rmse:.4f}")
        print(f"\n  Intensity Stats:")
        print(f"    True range : [{intensity_true.min():.3f}, {intensity_true.max():.3f}]")
        print(f"    Pred range : [{intensity_arr.min():.3f}, {intensity_arr.max():.3f}]")
        print(f"    True mean  : {intensity_true.mean():.3f}")
        print(f"    Pred mean  : {intensity_arr.mean():.3f}")

        if SHOW_CONFUSION_MATRICES:
            print("\n" + "=" * 70)
            print("  CONFUSION MATRICES")
            print("=" * 70)

            emotion_cm = confusion_matrix(all_emotion_labels, all_emotion_preds)
            print("\n  Emotion Confusion Matrix (rows=true, cols=pred):")
            print("       " + " ".join(f"{e[:4]:>5}" for e in EMOTIONS))
            for i, row in enumerate(emotion_cm):
                print(f"  {EMOTIONS[i][:4]:>4} " + " ".join(f"{x:5d}" for x in row))

            target_cm = confusion_matrix(all_target_labels, all_target_preds)
            print("\n  Target Confusion Matrix (rows=true, cols=pred):")
            print("       " + " ".join(f"{t[:4]:>5}" for t in TARGETS))
            for i, row in enumerate(target_cm):
                print(f"  {TARGETS[i][:4]:>4} " + " ".join(f"{x:5d}" for x in row))

    return metrics, (all_emotion_preds, all_emotion_labels,
                     all_target_preds,  all_target_labels,
                     all_intensity_preds, all_intensity_labels)

# ─── MAIN ────────────────────────────────────────────────────────────────────

def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"\n{'=' * 70}")
    print(f"  EMOTION CLASSIFIER TESTING")
    print(f"{'=' * 70}")
    print(f"  Device     : {device}")
    print(f"  Splits dir : {SPLITS_DIR}")
    print(f"  Model dir  : {MODEL_DIR}")
    print(f"  Targets    : {TARGETS}")
    print(f"{'=' * 70}\n")

    print("Loading test data...")
    test_samples = load_split(SPLITS_DIR, "test")
    print()

    print("Loading tokenizer...")
    tokenizer = RobertaTokenizer.from_pretrained(MODEL_DIR)
    print()

    print("Loading model...")
    model = EmotionMultiHeadModel("roberta-base", dropout=0.1).to(device)

    checkpoint_path = Path(MODEL_DIR) / "best_model.pt"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        epoch       = checkpoint.get("epoch", "unknown")
        val_metrics = checkpoint.get("val_metrics", {})
    elif "model_state" in checkpoint:
        model.load_state_dict(checkpoint["model_state"])
        epoch       = checkpoint.get("epoch", "unknown")
        val_metrics = checkpoint.get("val_metrics", {})
    else:
        model.load_state_dict(checkpoint)
        epoch, val_metrics = "unknown", {}

    print(f"  Loaded checkpoint from epoch {epoch}")
    if val_metrics:
        print(f"  Validation metrics at save:")
        print(f"    Loss         : {val_metrics.get('loss', 'N/A'):.4f}")
        print(f"    Emotion F1   : {val_metrics.get('emotion_f1', 'N/A'):.3f}")
        print(f"    Target F1    : {val_metrics.get('target_f1', 'N/A'):.3f}")
        print(f"    Intensity MAE: {val_metrics.get('intensity_mae', 'N/A'):.4f}")
    print()

    test_ds     = EmotionDataset(test_samples, tokenizer, MAX_LENGTH)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=2)

    loss_fn = MultiTaskLoss()

    print("Running evaluation on test set...")
    test_metrics, _ = evaluate(model, test_loader, loss_fn, device, detailed=True)

    print("\n" + "=" * 70)
    print("  TEST SET SUMMARY")
    print("=" * 70)
    print(f"  Total samples : {len(test_samples)}")
    print(f"  Loss          : {test_metrics['loss']:.4f}")
    print(f"  Emotion Acc   : {test_metrics['emotion_acc']:.3f} "
          f"({test_metrics['emotion_acc']*100:.1f}%)")
    print(f"  Emotion F1    : {test_metrics['emotion_f1']:.3f}")
    print(f"  Target Acc    : {test_metrics['target_acc']:.3f} "
          f"({test_metrics['target_acc']*100:.1f}%)")
    print(f"  Target F1     : {test_metrics['target_f1']:.3f}")
    print(f"  Intensity MAE : {test_metrics['intensity_mae']:.4f}")
    print(f"  Intensity RMSE: {test_metrics['intensity_rmse']:.4f}")

    print("\n  Per-class Emotion F1:")
    for i, f1 in enumerate(test_metrics["emotion_f1_per_class"]):
        print(f"    {EMOTIONS[i]:<12}: {f1:.4f}")

    print("\n  Per-class Target F1:")
    for i, f1 in enumerate(test_metrics["target_f1_per_class"]):
        print(f"    {TARGETS[i]:<12}: {f1:.4f}")

    print("=" * 70)

    if SAVE_RESULTS:
        results = {
            "test_metrics": {k: v for k, v in test_metrics.items()
                             if not k.endswith("_per_class")},
            "per_class_metrics": {
                "emotion_f1": {EMOTIONS[i]: test_metrics["emotion_f1_per_class"][i]
                               for i in range(len(EMOTIONS))},
                "target_f1":  {TARGETS[i]:  test_metrics["target_f1_per_class"][i]
                               for i in range(len(TARGETS))},
            },
            "checkpoint_info": {
                "epoch": epoch,
                "val_metrics_at_save": val_metrics,
            },
            "config": {
                "splits_dir":       str(SPLITS_DIR),
                "model_dir":        str(MODEL_DIR),
                "batch_size":       BATCH_SIZE,
                "max_length":       MAX_LENGTH,
                "num_test_samples": len(test_samples),
                "targets":          TARGETS,
            },
        }
        out = Path(MODEL_DIR) / "test_results.json"
        out.write_text(json.dumps(results, indent=2))
        print(f"\n✓ Results saved to {out}")


if __name__ == "__main__":
    main()


  EMOTION CLASSIFIER TESTING
  Device     : cuda
  Splits dir : /content/drive/MyDrive/emo/emotion_data/splits
  Model dir  : /content/drive/MyDrive/emo/finetune/checkpoints-2
  Targets    : ['you', 'other', 'self', 'situation']

Loading test data...
  Loaded  3741 samples from test.json

Loading tokenizer...

Loading model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded checkpoint from epoch 2
  Validation metrics at save:
    Loss         : 0.3888
    Emotion F1   : 0.947
    Target F1    : 0.931
    Intensity MAE: 0.1551

Running evaluation on test set...



  EMOTION CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger     0.8917    0.9088    0.9002       625
     sadness     0.9264    0.9279    0.9271       624
   happiness     0.9780    0.9984    0.9881       624
        fear     0.9731    0.9840    0.9785       624
     disgust     0.9372    0.9101    0.9235       623
    surprise     0.9917    0.9678    0.9796       621

    accuracy                         0.9495      3741
   macro avg     0.9497    0.9495    0.9495      3741
weighted avg     0.9496    0.9495    0.9495      3741


  TARGET CLASSIFICATION REPORT
              precision    recall  f1-score   support

         you     0.9863    0.9231    0.9536       936
       other     0.9608    0.9711    0.9659       934
        self     0.8979    0.9316    0.9144       935
   situation     0.8644    0.8782    0.8712       936

    accuracy                         0.9260      3741
   macro avg     0.9273    0.9260    0.9263      3741
weighted avg